In [1]:
class BlankObject:
    pass

In [2]:
# text_model_name = "indobenchmark/indobert-base-p2"
# # vision_model_name = "WinKawaks/vit-small-patch16-224"
# vision_model_name = "openai/clip-vit-base-patch32"

params = BlankObject()
params.text_model_name = "indobenchmark/indobert-base-p2"
# params.text_model_name = "bert-base-uncased"
# params.vision_model_name = "openai/clip-vit-base-patch32"
params.vision_model_name = "Galuh/clip-indonesian"
params.device = 1
params.simple_linear = False
params.text_size = 512
params.layers = 3
params.dropout_rate = 0.1
params.image_size = 768
params.num_train_epochs = 10
params.train_batch_size = 64
params.dev_batch_size = 64
params.max_len = 77
params.label_count = 2
params.output_dir = "./saved_models"
params.model = "sarcasm_model_v1"

# Dataset

In [3]:
import pandas as pd
import numpy as np
import random
from torch.utils.data import Dataset
import torch
import os
from PIL import Image
from pathlib import Path

from transformers import AutoTokenizer, AutoProcessor

In [4]:
cwd = Path.cwd()
dataset_dir = cwd / "dataset"

In [5]:
df = pd.read_json("./dataset/text_json_id/dataset_translated_fixed.json", orient="records", dtype={"image_id": str, "label": int}).set_index("image_id")
df.head()

,text,label,split,text_translated
image_id,,,,
840006160660983809,<user> thanks for showing up for our appointme...,1,train,<user> Terima kasih sudah datang ke janji temu...
908913372199915520,haha .,1,train,haha .
916496521406726145,i love waiting <num> min for a cab - such shor...,1,train,Aku suka menunggu <num> menit untuk taksi — be...
916364004129304576,22 super funny quotes <user>,1,train,22 kutipan yang super lucu <pengguna>
853866052589154304,goog morning,1,train,Selamat pagi


In [ ]:
# whitelist itu list gambar yang gk dominan teksnya

with open("./dataset/whitelist.txt", "r") as f:
    whitelist = set(line.strip()[:-4] for line in f)

df = df[df["image_id"].isin(whitelist)]

: 

In [ ]:
print(df.describe())
print(df.info())
print(df.value_counts(['split', 'label']))

In [6]:
class MMSD2_id_dataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe.to_dict(orient="index")
        self.image_ids=list(self.data.keys())
        for id in self.data.keys():
            self.data[id]["image_path"] = dataset_dir / "dataset_image" / f"{id}.jpg"

    def image_loader(self,id):
        return Image.open(self.data[id]["image_path"])
    
    def text_loader(self,id):
        return self.data[id]["text_translated"]

    def __getitem__(self, index):
        id=self.image_ids[index]
        text = self.text_loader(id)
        image_feature = self.image_loader(id)
        label = self.data[id]["label"]
        return text,image_feature, label, id

    def __len__(self):
        return len(self.image_ids)
    @staticmethod
    def collate_func(batch_data):
        batch_size = len(batch_data)
 
        if batch_size == 0:
            return {}

        text_list = []
        image_list = []
        label_list = []
        id_list = []
        for instance in batch_data:
            text_list.append(instance[0])
            image_list.append(instance[1])
            label_list.append(instance[2])
            id_list.append(instance[3])
        return text_list, image_list, label_list, id_list

### JALANIN INI KALO MAU CUSTOM SPLIT

In [7]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

In [8]:
# Get all indices and labels
indices = list(range(len(df)))
labels = df.label

# Split indices (stratify=labels ensures both sets have the same class proportions)
train_indices, val_test_indices = train_test_split(indices, test_size=0.4, stratify=labels)

labels_test_val = labels.iloc[val_test_indices].tolist()
val_indices, test_indices = train_test_split(val_test_indices, test_size=0.5, stratify=labels_test_val)

# Create virtual subsets using these indices
train_dataset_subset = Subset(df, train_indices)
val_dataset_subset = Subset(df, val_indices)
test_dataset_subset = Subset(df, test_indices)

In [9]:
print(df.iloc[train_dataset_subset.indices].value_counts(['split', 'label']))
print(df.iloc[val_dataset_subset.indices].value_counts(['split', 'label']))
print(df.iloc[test_dataset_subset.indices].value_counts(['split', 'label']))

split  label
train  0        6140
       1        5747
test   0         839
valid  0         809
test   1         647
valid  1         599
Name: count, dtype: int64
split  label
train  0        2045
       1        1944
valid  0         282
test   0         269
valid  1         211
test   1         176
Name: count, dtype: int64
split  label
train  0        2055
       1        1885
valid  0         277
test   0         264
valid  1         232
test   1         214
Name: count, dtype: int64


In [ ]:
train_dataset = MMSD2_id_dataset(df.iloc[train_dataset_subset.indices])
val_dataset = MMSD2_id_dataset(df.iloc[val_dataset_subset.indices])
test_dataset = MMSD2_id_dataset(df.iloc[test_dataset_subset.indices])

### JALANIN INI KALO MAU DEFAULT SPLIT NGIKUT UPSTREAM

In [10]:
train_dataset = MMSD2_id_dataset(df[df["split"] == "train"])
val_dataset = MMSD2_id_dataset(df[df["split"] == "valid"])
test_dataset = MMSD2_id_dataset(df[df["split"] == "test"])

# Model

In [17]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoProcessor, AutoModel, AlbertTokenizer, AlbertModel
from transformers import CLIPVisionModel, HybridCLIP
from torchinfo import summary
from torch import nn
from transformers import CLIPModel,BertConfig
from transformers.models.bert.modeling_bert import BertLayer
import copy
from sklearn import metrics

ImportError: cannot import name 'HybridCLIP' from 'transformers' (/root/projects/bukan-skripsi/notebooks/2nd_exp/.venv/lib/python3.12/site-packages/transformers/__init__.py)

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# torch.cuda.set_device(1)

In [13]:
class MultimodalEncoder(nn.Module):
    def __init__(self, config, layer_number):
        super(MultimodalEncoder, self).__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.hidden_size,       # 768
            nhead=config.num_attention_heads, # 12
            dim_feedforward=config.intermediate_size,  # 3072
            dropout=config.hidden_dropout_prob,
            activation='gelu',
            batch_first=True,                 # [B, seq, dim] convention
            norm_first=False
        )
        self.layer = nn.ModuleList([
            copy.deepcopy(encoder_layer) for _ in range(layer_number)
        ])

    def forward(self, hidden_states, attention_mask, output_all_encoded_layers=True):
        all_encoder_layers = []

        # nn.TransformerEncoderLayer expects mask as additive float [B, seq, seq]
        # or key_padding_mask as bool [B, seq]
        # Your attention_mask coming in is [B, 1, 1, seq] additive float — convert it:
        # squeeze to [B, seq], then invert to key_padding_mask (True = ignore)
        key_padding_mask = (attention_mask.squeeze(1).squeeze(1) == -10000.0)

        for layer_module in self.layer:
            hidden_states = layer_module(
                hidden_states,
                src_key_padding_mask=key_padding_mask
            )
            if output_all_encoded_layers:
                all_encoder_layers.append(hidden_states)

        if not output_all_encoded_layers:
            all_encoder_layers.append(hidden_states)

        return all_encoder_layers, None  # None = no attention weights returned
    
class SarcasmModel(nn.Module):
    def __init__(self, params):
        super(SarcasmModel, self).__init__()
        
        # 1. Decouple the encoders
        self.text_model = AutoModel.from_pretrained(params.text_model_name)
        self.vision_model = CLIPVisionModel.from_pretrained(params.vision_model_name)
        
        # 2. Use IndoBERT's config for the fusion transformer (768 dimensions)
        self.config = self.text_model.config
        self.config.num_hidden_layers = params.layers
        # self.config._attn_implementation = "eager"
        
        self.trans = MultimodalEncoder(self.config, layer_number=params.layers)
        
        if params.simple_linear:
            self.text_linear = nn.Linear(params.text_size, params.text_size)
            self.image_linear = nn.Linear(params.image_size, params.image_size)
        else:
            self.text_linear = nn.Sequential(
                nn.Linear(params.text_size, params.text_size),
                nn.Dropout(params.dropout_rate),
                nn.GELU()
            )
            self.image_linear = nn.Sequential(
                nn.Linear(params.image_size, params.image_size),
                nn.Dropout(params.dropout_rate),
                nn.GELU()
            )

        self.classifier_fuse = nn.Linear(params.text_size, params.label_count)
        self.classifier_text = nn.Linear(params.text_size, params.label_count)
        self.classifier_image = nn.Linear(params.image_size, params.label_count)

        self.loss_fct = nn.CrossEntropyLoss()
        self.att = nn.Linear(params.text_size, 1, bias=False)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        # 1. Independent Text Encoding (IndoBERT)
        text_outputs = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        text_features = text_outputs.last_hidden_state  # [B, L_text, 768]
        text_feature = text_outputs.pooler_output       # [B, 768]

        # 2. Independent Image Encoding (CLIP Vision)
        vision_outputs = self.vision_model(pixel_values=pixel_values)
        image_features = vision_outputs.last_hidden_state # [B, L_image, 768]
        image_feature = vision_outputs.pooler_output      # [B, 768]

        # Transform unimodal features for independent classification
        text_feature = self.text_linear(text_feature)
        image_feature = self.image_linear(image_feature)

        # 3. Multimodal Fusion Preparation
        # Directly concatenate the 768D sequences
        input_embeds = torch.cat((image_features, text_features), dim=1)
        
        # Dynamic sequence length for images
        vision_seq_len = image_features.shape[1]
        vision_mask = torch.ones(text_features.shape[0], vision_seq_len, device=text_features.device)
        concat_mask = torch.cat((vision_mask, attention_mask), dim=-1)
        
        extended_attention_mask = concat_mask.unsqueeze(1).unsqueeze(2)
        extended_attention_mask = extended_attention_mask.to(dtype=next(self.parameters()).dtype)
        extended_attention_mask = (1.0 - extended_attention_mask) * -10000.0

        # 4. Multimodal Fusion Pass
        combined_seq_len = input_embeds.shape[1]
        mask_seq_len = concat_mask.shape[-1]
        
        fuse_hiddens, all_attentions = self.trans(input_embeds, extended_attention_mask, output_all_encoded_layers=False)
        fuse_hiddens = fuse_hiddens[-1]
        
        # Extract the fused text and image representations
        new_text_features = fuse_hiddens[:, vision_seq_len:, :]
        new_text_feature = new_text_features[
            torch.arange(new_text_features.shape[0], device=input_ids.device), 
            input_ids.to(torch.int).argmax(dim=-1)
        ]
        
        # Extract image [CLS] token
        new_image_feature = fuse_hiddens[:, 0, :]

        # 5. Attention-based Weighting
        text_weight = self.att(new_text_feature)
        image_weight = self.att(new_image_feature)    
        att = nn.functional.softmax(torch.stack((text_weight, image_weight), dim=-1), dim=-1)
        tw, iw = att.split([1,1], dim=-1)
        fuse_feature = tw.squeeze(1) * new_text_feature + iw.squeeze(1) * new_image_feature

        # 6. Classification
        logits_fuse = self.classifier_fuse(fuse_feature)
        logits_text = self.classifier_text(text_feature)
        logits_image = self.classifier_image(image_feature)
   
        fuse_score = nn.functional.softmax(logits_fuse, dim=-1)
        text_score = nn.functional.softmax(logits_text, dim=-1)
        image_score = nn.functional.softmax(logits_image, dim=-1)

        # Average the scores for a valid probability distribution
        score = (fuse_score + text_score + image_score) / 3.0

        outputs = (score,)
        if labels is not None:
            loss_fuse = self.loss_fct(logits_fuse, labels)
            loss_text = self.loss_fct(logits_text, labels)
            loss_image = self.loss_fct(logits_image, labels)
            loss = loss_fuse + loss_text + loss_image

            outputs = (loss,) + outputs
            
        return outputs

# Train

In [11]:
from tqdm import tqdm, trange

In [14]:
train_loader = DataLoader(dataset=train_dataset,
                              batch_size=params.train_batch_size,
                              collate_fn=MMSD2_id_dataset.collate_func,
                              shuffle=True)

model = SarcasmModel(params).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/4.49k [00:00<?, ?B/s]

OSError: Galuh/clip-indonesian does not appear to have a file named pytorch_model.bin or model.safetensors.

In [24]:
del model
torch.cuda.empty_cache()

In [26]:
# clip_params = list(map(id, model.parameters()))
# base_params = filter(lambda p: id(p) not in clip_params, model.parameters())
# optimizer = ada([
#         {"params": base_params},
#         {"params": model.model.parameters(),"lr": args.clip_learning_rate}
#         ], lr=args.learning_rate, weight_decay=args.weight_decay)
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
# scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(args.warmup_proportion * total_steps),
#                                         num_training_steps=total_steps)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
processor = AutoProcessor.from_pretrained(params.vision_model_name)
tokenizer = AutoTokenizer.from_pretrained(params.text_model_name)

In [ ]:
# # 1. See what the model expects for dimensions
# print(f"Expected size: {processor.image_processor.size}") 

# # 2. See normalization values (Mean and Std)
# print(f"Mean: {processor.image_processor.image_mean}")
# print(f"Std: {processor.image_processor.image_std}")

# # 3. Check the output shape with a dummy image (e.g., a random tensor)
# import torch
# dummy_image = torch.randn(3, 500, 500) # Simulating a 500x500 RGB image
# # Rescale dummy_image to [0,1], convert to PIL, then process
# img = dummy_image.clone().cpu()
# img = (img - img.min()) / (img.max() - img.min() + 1e-8)
# np_img = (img.permute(1, 2, 0).mul(255).to(torch.uint8).numpy())
# pil_img = Image.fromarray(np_img)
# pixel_values = processor(images=pil_img, return_tensors="pt")["pixel_values"]

# print(f"Processed image shape: {pixel_values.shape}")
# # Likely [1, 3, 224, 224]

In [21]:
def evaluate_acc_f1(params, model, device, data, processor, macro=False,pre = None, mode='test'):
        data_loader = DataLoader(data, batch_size=params.dev_batch_size, collate_fn=MMSD2_id_dataset.collate_func,shuffle=False)
        n_correct, n_total = 0, 0
        t_targets_all, t_outputs_all = None, None

        model.eval()
        sum_loss = 0.
        sum_step = 0
        with torch.no_grad():
            for i_batch, t_batch in enumerate(data_loader):
                text_list, image_list, label_list, id_list = t_batch
                text_inputs = tokenizer(text_list, padding='max_length', truncation=True, max_length=params.max_len, return_tensors="pt").to(device)
                image_inputs = processor(images=image_list, return_tensors="pt").to(device)
                labels = torch.tensor(label_list).to(device)

                input_ids = text_inputs['input_ids']
                attention_mask = text_inputs['attention_mask']
                pixel_values = image_inputs['pixel_values']
                
                t_targets = labels
                loss, t_outputs = model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values, labels=labels)
                sum_loss += loss.item()
                sum_step += 1
  
                outputs = torch.argmax(t_outputs, -1)

                n_correct += (outputs == t_targets).sum().item()
                n_total += len(outputs)

                if t_targets_all is None:
                    t_targets_all = t_targets
                    t_outputs_all = outputs
                else:
                    t_targets_all = torch.cat((t_targets_all, t_targets), dim=0)
                    t_outputs_all = torch.cat((t_outputs_all, outputs), dim=0)
        if mode == 'test':
            # wandb.log({'test_loss': sum_loss/sum_step})
            print(f"Test Loss: {sum_loss/sum_step}")
        else:
            # wandb.log({'dev_loss': sum_loss/sum_step})
            print(f"Dev Loss: {sum_loss/sum_step}")
        if pre != None:
            with open(pre,'w',encoding='utf-8') as fout:
                predict = t_outputs_all.cpu().numpy().tolist()
                label = t_targets_all.cpu().numpy().tolist()
                for x,y,z in zip(predict,label):
                    fout.write(str(x) + str(y) +z+ '\n')
        if not macro:   
            acc = n_correct / n_total
            f1 = metrics.f1_score(t_targets_all.cpu(), t_outputs_all.cpu())
            precision =  metrics.precision_score(t_targets_all.cpu(),t_outputs_all.cpu())
            recall = metrics.recall_score(t_targets_all.cpu(),t_outputs_all.cpu())
        else:
            acc = n_correct / n_total
            f1 = metrics.f1_score(t_targets_all.cpu(), t_outputs_all.cpu(), labels=[0, 1],average='macro')
            precision =  metrics.precision_score(t_targets_all.cpu(),t_outputs_all.cpu(), labels=[0, 1],average='macro')
            recall = metrics.recall_score(t_targets_all.cpu(),t_outputs_all.cpu(), labels=[0, 1],average='macro')
        return acc, f1 ,precision,recall

In [27]:
max_acc = 0.
for i_epoch in trange(0, int(params.num_train_epochs), desc="Epoch", disable=False):
    sum_loss = 0.
    sum_step = 0

    iter_bar = tqdm(train_loader, desc="Iter (loss=X.XXX)", disable=False)
    model.train()

    for step, batch in enumerate(iter_bar):
        text_list, image_list, label_list, id_list = batch

        text_inputs = tokenizer(text_list, padding='max_length', truncation=True, max_length=params.max_len, return_tensors="pt").to(device)
        image_inputs = processor(images=image_list, return_tensors="pt").to(device)
        labels = torch.tensor(label_list).to(device)

        input_ids = text_inputs['input_ids']
        attention_mask = text_inputs['attention_mask']
        pixel_values = image_inputs['pixel_values']

        # Ensure text-related modules match the text encoder hidden size (fix 768 vs 512 mismatch)
        text_hid = model.text_model.config.hidden_size
        # determine current in_features of text_linear
        if isinstance(model.text_linear, nn.Sequential):
            cur_in = model.text_linear[0].in_features
        else:
            cur_in = model.text_linear.in_features
        if cur_in != text_hid:
            if params.simple_linear:
                model.text_linear = nn.Linear(text_hid, text_hid).to(device)
            else:
                model.text_linear = nn.Sequential(
                    nn.Linear(text_hid, text_hid),
                    nn.Dropout(params.dropout_rate),
                    nn.GELU()
                ).to(device)
                model.classifier_fuse = nn.Linear(text_hid, params.label_count).to(device)
                model.classifier_text = nn.Linear(text_hid, params.label_count).to(device)
                model.att = nn.Linear(text_hid, 1, bias=False).to(device)
                # recreate optimizer so it includes newly created params
            optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

        loss, score = model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values, labels=labels)
        sum_loss += loss.item()
        sum_step += 1

        iter_bar.set_description("Iter (loss=%5.3f)" % loss.item())
        loss.backward()
        optimizer.step()
        # if params.optimizer_name == 'adam':
        #     scheduler.step() 
        optimizer.zero_grad()

    print(f"Epoch {i_epoch} completed. Average Loss: {sum_loss/sum_step:.4f}")
    # wandb.log({'train_loss': sum_loss/sum_step})

    dev_acc, dev_f1 ,dev_precision,dev_recall = evaluate_acc_f1(params, model, device, val_dataset, processor, mode='dev')
    print(f"Epoch {i_epoch} completed. Dev Acc: {dev_acc:.4f}, Dev F1: {dev_f1:.4f}, Dev Precision: {dev_precision:.4f}, Dev Recall: {dev_recall:.4f}")
    # wandb.log({'dev_acc': dev_acc, 'dev_f1': dev_f1, 'dev_precision': dev_precision, 'dev_recall': dev_recall})
    # logging.info("i_epoch is {}, dev_acc is {}, dev_f1 is {}, dev_precision is {}, dev_recall is {}".format(i_epoch, dev_acc, dev_f1, dev_precision, dev_recall))

    if dev_acc > max_acc:
        max_acc = dev_acc

        path_to_save = os.path.join(params.output_dir, params.model)
        if not os.path.exists(path_to_save):
            os.mkdir(path_to_save)
        model_to_save = (model.module if hasattr(model, "module") else model)
        torch.save(model_to_save.state_dict(), os.path.join(path_to_save, 'model.pt'))

        test_acc, test_f1,test_precision,test_recall = evaluate_acc_f1(params, model, device, test_dataset, processor,macro = True, mode='test')
        _, test_f1_,test_precision_,test_recall_ = evaluate_acc_f1(params, model, device, test_dataset, processor, mode='test')
        # wandb.log({'test_acc': test_acc, 'macro_test_f1': test_f1,
        #             'macro_test_precision': test_precision,'macro_test_recall': test_recall, 'micro_test_f1': test_f1_,
        #             'micro_test_precision': test_precision_,'micro_test_recall': test_recall_})
        print(f"Epoch {i_epoch} completed. Test Acc: {test_acc:.4f}, Test F1: {test_f1:.4f}, Test Precision: {test_precision:.4f}, Test Recall: {test_recall:.4f}")
        print(f"Epoch {i_epoch} completed. Micro Test F1: {test_f1_:.4f}, Micro Test Precision: {test_precision_:.4f}, Micro Test Recall: {test_recall_:.4f}")

    torch.cuda.empty_cache()

Iter (loss=1.745): 100%|██████████| 310/310 [04:26<00:00,  1.16it/s]


Epoch 0 completed. Average Loss: 1.7128
Dev Loss: 1.68411974530471
Epoch 0 completed. Dev Acc: 0.7722, Dev F1: 0.7508, Dev Precision: 0.7123, Dev Recall: 0.7937
Test Loss: 1.6754049031358016


Epoch:  10%|█         | 1/10 [05:39<50:59, 339.96s/it]

Test Loss: 1.6754049031358016
Epoch 0 completed. Test Acc: 0.7609, Test F1: 0.7594, Test Precision: 0.7597, Test Recall: 0.7648
Epoch 0 completed. Micro Test F1: 0.7405, Micro Test Precision: 0.6948, Micro Test Recall: 0.7927


Iter (loss=1.114): 100%|██████████| 310/310 [04:26<00:00,  1.16it/s]


...

Dev Loss: 3.027518397883365
Epoch 8 completed. Dev Acc: 0.7369, Dev F1: 0.6946, Dev Precision: 0.6973, Dev Recall: 0.6919


Iter (loss=0.710): 100%|██████████| 310/310 [04:26<00:00,  1.17it/s]


Epoch 9 completed. Average Loss: 0.5233


Epoch: 100%|██████████| 10/10 [49:15<00:00, 295.52s/it]

Dev Loss: 3.3480490759799353
Epoch 9 completed. Dev Acc: 0.7307, Dev F1: 0.7067, Dev Precision: 0.6678, Dev Recall: 0.7505
